In [1]:
import sys, os, warnings
sys.path.append("../pipelines")
from aq import *
import great_expectations as gx
warnings.filterwarnings("ignore")

COUCHE = os.environ.get("COUCHE", "BRONZE")
DOSSIER = "/home/jovyan/work/reports" if os.path.isdir("/home/jovyan/work/reports") else "reports"
os.makedirs(f"{DOSSIER}/ge", exist_ok=True)

CHARGEMENT = {
 "BRONZE": {
  "crm_client":   ("client",   "SELECT client_id AS cle, client_id, 'crm' AS source_systeme, type_personne, telephone, cp, pays FROM bronze.crm_client WHERE _batch_id = :b"),
  "crm_contrat":  ("contrat",  "SELECT contrat_id AS cle, contrat_id, souscripteur_id, date_debut, date_fin, duree_garantie_mois FROM bronze.crm_contrat WHERE _batch_id = :b"),
  "crm_sinistre": ("sinistre", "SELECT sinistre_id AS cle, sinistre_id FROM bronze.crm_sinistre WHERE _batch_id = :b"),
  "cif_clients":  ("cif",      "SELECT column0 AS cle FROM bronze.cif_clients WHERE _batch_id = :b")},
 "SILVER": {
  "client":   ("client",   "SELECT CONCAT(source_systeme, '|', source_id) AS cle, source_id AS client_id, source_systeme, type_personne, telephone_normalise AS telephone, cp, pays_iso AS pays FROM silver.client"),
  "contrat":  ("contrat",  "SELECT contrat_id AS cle, contrat_id, souscripteur_id, date_debut, date_fin, duree_garantie_mois FROM silver.contrat WHERE :b = :b"),
  "sinistre": ("sinistre", "SELECT sinistre_id AS cle, sinistre_id FROM silver.sinistre WHERE :b = :b")}}
NAISSANCES = {"BRONZE": "SELECT client_id, date_naissance FROM bronze.crm_client WHERE _batch_id = :b",
              "SILVER": "SELECT source_id AS client_id, date_naissance FROM silver.client WHERE source_systeme = 'crm' AND :b = :b"}

def age_a(date, naissance):
    a = date.dt.year - naissance.dt.year
    avant = (date.dt.month < naissance.dt.month) | ((date.dt.month == naissance.dt.month) & (date.dt.day < naissance.dt.day))
    return a - avant.astype(int)

def charger(table):
    nature, requete = CHARGEMENT[COUCHE][table]
    df = lire(requete, b=BATCH)
    if nature == "contrat":
        for c in ["date_debut", "date_fin"]:
            df[c] = pd.to_datetime(df[c], errors="coerce")
        df["duree_garantie_mois"] = pd.to_numeric(df.duree_garantie_mois, errors="coerce")
        nais = lire(NAISSANCES[COUCHE], b=BATCH).drop_duplicates("client_id")
        nais["date_naissance"] = pd.to_datetime(nais.date_naissance, errors="coerce")
        df = df.merge(nais, how="left", left_on="souscripteur_id", right_on="client_id", suffixes=("", "_s"))
        df["age_debut"] = age_a(df.date_debut, df.date_naissance.fillna(df.date_debut))
        df.loc[df.date_naissance.isna(), "age_debut"] = float("nan")
    return nature, df

DONNEES = {table: charger(table) for table in CHARGEMENT[COUCHE]}
for t, (nature, d) in DONNEES.items():
    print(f"{COUCHE.lower()}.{t:<13} {len(d):>7} lignes")
REGLES = lire("SELECT rule_id, severite, seuil_min FROM dq.regle").set_index("rule_id")
CODES_PAYS = lire("SELECT code_pays FROM silver.ref_pays").code_pays.tolist()
def mostly(rule_id):
    return float(REGLES.loc[rule_id, "seuil_min"]) / 100

bronze.crm_client     100000 lignes
bronze.crm_contrat    100000 lignes
bronze.crm_sinistre   100000 lignes
bronze.cif_clients     40000 lignes


In [4]:
context = gx.get_context(mode="file", project_root_dir=f"{DOSSIER}/ge")
source = context.sources.add_or_update_pandas(name="mariadb_df")

def asset(nom):
    try:
        return source.get_asset(nom)
    except LookupError:
        return source.add_dataframe_asset(name=nom)

def valider(table, nature, df, ajouter):
    br = asset(table).build_batch_request(dataframe=df)
    suite = f"assurconnect_{COUCHE.lower()}_{table}_suite"
    context.add_or_update_expectation_suite(suite)
    v = context.get_validator(batch_request=br, expectation_suite_name=suite)
    ajouter(v)
    v.save_expectation_suite(discard_failed_expectations=False)
    cp = context.add_or_update_checkpoint(name=f"dq_{COUCHE.lower()}_{table}", validations=[{"batch_request": br, "expectation_suite_name": suite}])
    return list(cp.run().run_results.values())[0]["validation_result"].results

In [5]:
SOUSCRIPTEUR_CRM = dict(row_condition='source_systeme == "crm" and type_personne == "souscripteur"', condition_parser="pandas")
FRANCE_CRM       = dict(row_condition='source_systeme == "crm" and pays == "FR"', condition_parser="pandas")
CRM              = dict(row_condition='source_systeme == "crm"', condition_parser="pandas")

def regles_client(v):
    v.expect_column_values_to_not_be_null(column="telephone", mostly=mostly("RQ-003"), meta={"rule_id": "RQ-003"}, **SOUSCRIPTEUR_CRM)
    v.expect_column_values_to_match_regex(column="telephone", regex=r"\S", mostly=mostly("RQ-003"), meta={"rule_id": "RQ-003"}, **SOUSCRIPTEUR_CRM)
    v.expect_column_values_to_match_regex(column="cp", regex=r"^\d{5}$", mostly=mostly("RQ-005"), meta={"rule_id": "RQ-005", "nuls_en_echec": True}, **FRANCE_CRM)
    v.expect_column_values_to_be_in_set(column="pays", value_set=CODES_PAYS, meta={"rule_id": "RQ-007", "nuls_en_echec": True}, **CRM)
    v.expect_column_values_to_be_in_set(column="type_personne", value_set=["prospect", "souscripteur", "assure"], meta={"rule_id": "RQ-009", "nuls_en_echec": True}, **CRM)

def regles_contrat(v):
    v.expect_column_pair_values_A_to_be_greater_than_B(column_A="date_fin", column_B="date_debut", or_equal=False, meta={"rule_id": "RQ-001"})
    v.expect_column_values_to_be_between(column="age_debut", min_value=18, meta={"rule_id": "RQ-002"})
    v.expect_column_values_to_be_in_set(column="duree_garantie_mois", value_set=[12, 24, 36, 60], meta={"rule_id": "RQ-010", "nuls_en_echec": True})

def cle_primaire(v):
    v.expect_column_values_to_not_be_null(column="cle", meta={"rule_id": "RQ-020"})
    v.expect_column_values_to_be_unique(column="cle", meta={"rule_id": "RQ-020"})

SUITES = {"client": [regles_client, cle_primaire], "contrat": [regles_contrat, cle_primaire], "sinistre": [cle_primaire], "cif": [cle_primaire]}

def tout_ajouter(nature):
    return lambda v: [f(v) for f in SUITES[nature]]

mesures_ge = []
for table, (nature, df) in DONNEES.items():
    resultats = valider(table, nature, df, tout_ajouter(nature))
    par_regle = {}
    for r in resultats:
        m = r.expectation_config.meta
        n, ko = r.result["element_count"], r.result["unexpected_count"] + (r.result.get("missing_count", 0) if m.get("nuls_en_echec") else 0)
        t0, k0 = par_regle.get(m["rule_id"], (n, 0))
        par_regle[m["rule_id"]] = (t0, k0 + ko)
    for rule, (n, ko) in par_regle.items():
        mesures_ge.append({"rule_id": rule, "table": table, "testees": int(n), "echecs": int(ko)})
bilan = pd.DataFrame(mesures_ge).sort_values(["rule_id", "table"])
bilan["conformite_pct"] = (100 * (bilan.testees - bilan.echecs) / bilan.testees).round(3)
bilan["seuil_pct"] = [float(REGLES.loc[r, "seuil_min"]) for r in bilan.rule_id]
bilan["severite"] = [REGLES.loc[r, "severite"] for r in bilan.rule_id]
bilan["verdict"] = ["conforme" if c >= s else "sous le seuil" for c, s in zip(bilan.conformite_pct, bilan.seuil_pct)]
print(bilan.to_string(index=False))
context.build_data_docs()

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

rule_id        table  testees  echecs  conformite_pct  seuil_pct      severite       verdict
 RQ-001  crm_contrat   100000     800          99.200      100.0     BLOQUANTE sous le seuil
 RQ-002  crm_contrat   100000     301          99.699      100.0     BLOQUANTE sous le seuil
 RQ-003   crm_client    49898    5989          87.998       98.0 AVERTISSEMENT sous le seuil
 RQ-005   crm_client    95437    1456          98.474       98.0 AVERTISSEMENT      conforme
 RQ-007   crm_client   100000    3050          96.950      100.0     BLOQUANTE sous le seuil
 RQ-009   crm_client   100000     200          99.800      100.0     BLOQUANTE sous le seuil
 RQ-010  crm_contrat   100000     600          99.400      100.0     BLOQUANTE sous le seuil
 RQ-020  cif_clients    40000       0         100.000      100.0     BLOQUANTE      conforme
 RQ-020   crm_client   100000       0         100.000      100.0     BLOQUANTE      conforme
 RQ-020  crm_contrat   100000       0         100.000      100.0     B

{'local_site': 'file:///home/jovyan/work/reports/ge/gx/uncommitted/data_docs/local_site/index.html'}

In [6]:
TYPES = {
    "RQ-001": "expect_column_pair_values_A_to_be_greater_than_B",
    "RQ-002": "expect_column_values_to_be_between (min 18)",
    "RQ-003": "expect_column_values_to_not_be_null + expect_column_values_to_match_regex (\\S)",
    "RQ-005": "expect_column_values_to_match_regex (^\\d{5}$), adresses FR",
    "RQ-007": "expect_column_values_to_be_in_set (silver.ref_pays)",
    "RQ-009": "expect_column_values_to_be_in_set",
    "RQ-010": "expect_column_values_to_be_in_set (12, 24, 36, 60)",
    "RQ-020": "expect_column_values_to_not_be_null + expect_column_values_to_be_unique",
}
fmt = lambda n: f"{int(n):,}".replace(",", " ")
lignes_md = ["| Règle | Table bronze | Expectation (type) | Lignes testées | Lignes en échec | Taux de réussite | Sévérité retenue (bloquante / avertissement) |",
             "|---|---|---|---|---|---|---|"]
for r in bilan.itertuples(index=False):
    lignes_md.append(f"| {r.rule_id} | {r.table} | {TYPES[r.rule_id]} | {fmt(r.testees)} | {fmt(r.echecs)} | {r.conformite_pct:.3f} % | {r.severite.lower()} (seuil {r.seuil_pct:g} %) |")
print("\n".join(lignes_md))

| Règle | Table bronze | Expectation (type) | Lignes testées | Lignes en échec | Taux de réussite | Sévérité retenue (bloquante / avertissement) |
|---|---|---|---|---|---|---|
| RQ-001 | crm_contrat | expect_column_pair_values_A_to_be_greater_than_B | 100 000 | 800 | 99.200 % | bloquante (seuil 100 %) |
| RQ-002 | crm_contrat | expect_column_values_to_be_between (min 18) | 100 000 | 301 | 99.699 % | bloquante (seuil 100 %) |
| RQ-003 | crm_client | expect_column_values_to_not_be_null + expect_column_values_to_match_regex (\S) | 49 898 | 5 989 | 87.998 % | avertissement (seuil 98 %) |
| RQ-005 | crm_client | expect_column_values_to_match_regex (^\d{5}$), adresses FR | 95 437 | 1 456 | 98.474 % | avertissement (seuil 98 %) |
| RQ-007 | crm_client | expect_column_values_to_be_in_set (silver.ref_pays) | 100 000 | 3 050 | 96.950 % | bloquante (seuil 100 %) |
| RQ-009 | crm_client | expect_column_values_to_be_in_set | 100 000 | 200 | 99.800 % | bloquante (seuil 100 %) |
| RQ-010 | crm_contr

In [7]:
cl = DONNEES["crm_client"][1]
sous = cl[cl.type_personne == "souscripteur"]
tel = sous.telephone
rq003_null = int(tel.isna().sum())
rq003_tout = int((tel.isna() | (tel.fillna("x").str.strip() == "")).sum())

cp_tous = int((~cl.cp.fillna("").str.fullmatch(r"\d{5}")).sum())
fr = cl[cl.pays == "FR"]
cp_fr = int((~fr.cp.fillna("").str.fullmatch(r"\d{5}")).sum())

import unicodedata
sans_accent = lambda s: "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")
domaine = {"prospect", "souscripteur", "assure"}
tp = cl.type_personne
rq009_exact = int((~tp.isin(domaine)).sum())
rq009_souple = int((~tp.fillna("").map(lambda s: sans_accent(s.strip().lower())).isin(domaine)).sum())

pieges = pd.DataFrame([
    ("RQ-003", "nul, vide et blanc : seul NULL compté", "NULL seulement", rq003_null, "NULL + vide + blanc", rq003_tout),
    ("RQ-005", "adresses étrangères testées comme françaises", "toutes les adresses", cp_tous, "adresses FR seulement", cp_fr),
    ("RQ-009", "comparaison non exacte (casse, accents, espaces)", "comparaison souple", rq009_souple, "comparaison exacte", rq009_exact),
], columns=["regle", "piege", "avant", "chiffre_avant", "apres", "chiffre_apres"])
print(pieges.to_string(index=False))
print()
print("| Règle | Piège | Chiffre avant correction | Chiffre après correction | Preuve |")
print("|---|---|---|---|---|")
for r in pieges.itertuples(index=False):
    print(f"| {r.regle} | {r.piege} | {fmt(r.chiffre_avant)} ({r.avant}) | {fmt(r.chiffre_apres)} ({r.apres}) | `notebooks/S2_Fatine.ipynb`, cellule 5 |")

 regle                                            piege               avant  chiffre_avant                 apres  chiffre_apres
RQ-003            nul, vide et blanc : seul NULL compté      NULL seulement           1995   NULL + vide + blanc           5989
RQ-005     adresses étrangères testées comme françaises toutes les adresses           2317 adresses FR seulement           1456
RQ-009 comparaison non exacte (casse, accents, espaces)  comparaison souple             80    comparaison exacte            200

| Règle | Piège | Chiffre avant correction | Chiffre après correction | Preuve |
|---|---|---|---|---|
| RQ-003 | nul, vide et blanc : seul NULL compté | 1 995 (NULL seulement) | 5 989 (NULL + vide + blanc) | `notebooks/S2_Fatine.ipynb`, cellule 5 |
| RQ-005 | adresses étrangères testées comme françaises | 2 317 (toutes les adresses) | 1 456 (adresses FR seulement) | `notebooks/S2_Fatine.ipynb`, cellule 5 |
| RQ-009 | comparaison non exacte (casse, accents, espaces) | 80 (comparaiso

In [8]:
COMPLET = lambda ident: {"result_format": "COMPLETE", "unexpected_index_column_names": [ident]}

def lignes_en_echec(table, ident, expectation, **kwargs):
    nature, df = DONNEES[table]
    context.add_or_update_expectation_suite("assurconnect_echecs")
    v = context.get_validator(batch_request=asset(f"echecs_{table}").build_batch_request(dataframe=df), expectation_suite_name="assurconnect_echecs")
    return getattr(v, expectation)(result_format=COMPLET(ident), **kwargs).result.get("unexpected_index_list", [])

echecs = []
if COUCHE == "BRONZE":
    t = "crm_client"
    for rule, colonne, motif_txt, exp, kw in [
        ("RQ-003", "telephone", "téléphone nul, vide ou blanc", "expect_column_values_to_not_be_null", SOUSCRIPTEUR_CRM),
        ("RQ-003", "telephone", "téléphone nul, vide ou blanc", "expect_column_values_to_match_regex", dict(regex=r"\S", **SOUSCRIPTEUR_CRM)),
        ("RQ-005", "cp", "code postal : pas 5 chiffres", "expect_column_values_to_match_regex", dict(regex=r"^\d{5}$", **FRANCE_CRM)),
        ("RQ-009", "type_personne", "type_personne hors domaine", "expect_column_values_to_be_in_set", dict(value_set=["prospect", "souscripteur", "assure"], **CRM))]:
        for ligne in lignes_en_echec(t, "client_id", exp, column=colonne, **kw):
            echecs.append({"rule_id": rule, "table": t, "id": ligne["client_id"], "valeur": ligne.get(colonne), "motif": motif_txt})
    t = "crm_contrat"
    for rule, colonne, motif_txt, exp, kw in [("RQ-010", "duree_garantie_mois", "durée de garantie hors liste", "expect_column_values_to_be_in_set", dict(value_set=[12, 24, 36, 60]))]:
        for ligne in lignes_en_echec(t, "contrat_id", exp, column=colonne, **kw):
            echecs.append({"rule_id": rule, "table": t, "id": ligne["contrat_id"], "valeur": ligne.get(colonne), "motif": motif_txt})
    for ligne in lignes_en_echec("crm_contrat", "contrat_id", "expect_column_pair_values_A_to_be_greater_than_B", column_A="date_fin", column_B="date_debut", or_equal=False):
        echecs.append({"rule_id": "RQ-001", "table": "crm_contrat", "id": ligne["contrat_id"], "valeur": f"{ligne.get('date_fin')} <= {ligne.get('date_debut')}", "motif": "date de fin non postérieure au début"})
if COUCHE == "BRONZE":
    echecs = pd.DataFrame(echecs)
    os.makedirs(f"{DOSSIER}/echecs", exist_ok=True)
    for rule, sous in echecs.groupby("rule_id"):
        sous.to_csv(f"{DOSSIER}/echecs/lignes_en_echec_{rule}.csv", index=False)
    print(echecs.groupby(["rule_id", "table"]).size().rename("lignes_en_echec"))
    print("\nÉchantillon de 20 lignes à commenter (RQ-009) :\n", echecs[echecs.rule_id == "RQ-009"].sample(20, random_state=1).to_string(index=False))
    print("\nMotifs distincts de RQ-009, avec leur fréquence :\n", echecs[echecs.rule_id == "RQ-009"].valeur.value_counts())

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

rule_id  table      
RQ-001   crm_contrat     800
RQ-003   crm_client     5989
RQ-005   crm_client     1456
RQ-009   crm_client      200
RQ-010   crm_contrat     600
Name: lignes_en_echec, dtype: int64

Échantillon de 20 lignes à commenter (RQ-009) :
 rule_id      table      id       valeur                      motif
 RQ-009 crm_client A-34062       assuré type_personne hors domaine
 RQ-009 crm_client A-25815 Souscripteur type_personne hors domaine
 RQ-009 crm_client A-23998       assuré type_personne hors domaine
 RQ-009 crm_client H-18891       assuré type_personne hors domaine
 RQ-009 crm_client  S-4573       assuré type_personne hors domaine
 RQ-009 crm_client  S-9808       client type_personne hors domaine
 RQ-009 crm_client H-17183 Souscripteur type_personne hors domaine
 RQ-009 crm_client A-12289       assuré type_personne hors domaine
 RQ-009 crm_client A-22145       client type_personne hors domaine
 RQ-009 crm_client S-21356       client type_personne hors domaine
 RQ-009 crm

In [9]:
def famille(regle, valeur):
    v = "" if valeur is None or (isinstance(valeur, float) and pd.isna(valeur)) else str(valeur)
    if regle == "RQ-003":
        return "téléphone NULL" if valeur is None or v == "nan" else ("téléphone vide" if v == "" else "téléphone blanc (espaces)")
    if regle == "RQ-005":
        if v == "" or v == "None":
            return "code postal absent"
        if v.isdigit() and len(v) == 4:
            return "zéro initial perdu (4 chiffres)"
        return "format non conforme"
    if regle == "RQ-009":
        if v in ("", "None"):
            return "type non renseigné"
        if sans_accent(v.strip().lower()) in domaine:
            return "casse ou accent différents"
        return "valeur hors domaine"
    if regle == "RQ-010":
        return f"durée hors liste ({v})"
    if regle == "RQ-001":
        fin, debut = [x.strip() for x in v.split("<=")]
        return "date de fin égale au début" if fin == debut else "date de fin antérieure au début"
    return "autre"

echecs["famille"] = [famille(r, v) for r, v in zip(echecs.rule_id, echecs.valeur)]
print("Familles d'erreur sur l'ensemble des lignes en échec :")
print(echecs.groupby(["rule_id", "famille"]).size().rename("lignes").to_string())

parts = []
regles_echec = echecs.rule_id.unique().tolist()
par_regle = max(1, 20 // len(regles_echec))
for r in regles_echec:
    sous_e = echecs[echecs.rule_id == r]
    parts.append(sous_e.sample(min(par_regle, len(sous_e)), random_state=1))
echantillon = pd.concat(parts)
reste = echecs.drop(echantillon.index)
if len(echantillon) < 20 and len(reste):
    echantillon = pd.concat([echantillon, reste.sample(min(20 - len(echantillon), len(reste)), random_state=1)])
echantillon = echantillon.head(20).sort_values(["rule_id", "famille"])
print("\nÉchantillon de 20 lignes :")
print(echantillon[["rule_id", "table", "id", "valeur", "famille"]].to_string(index=False))

with open(f"{DOSSIER}/S2_echantillon_20_lignes_en_echec.md", "w", encoding="utf-8") as f:
    f.write("| Règle | Table | Identifiant | Valeur | Famille d'erreur |\n|---|---|---|---|---|\n")
    for r in echantillon.itertuples(index=False):
        f.write(f"| {r.rule_id} | {r.table} | {r.id} | {r.valeur} | {r.famille} |\n")
print("\nNombre de lignes de l'échantillon par famille :")
print(echantillon.groupby(["rule_id", "famille"]).size().rename("lignes").to_string())

Familles d'erreur sur l'ensemble des lignes en échec :
rule_id  famille                        
RQ-001   date de fin antérieure au début     400
         date de fin égale au début          400
RQ-003   téléphone NULL                     1995
         téléphone blanc (espaces)          1999
         téléphone vide                     1995
RQ-005   zéro initial perdu (4 chiffres)    1456
RQ-009   casse ou accent différents          120
         valeur hors domaine                  80
RQ-010   durée hors liste (120)               55
         durée hors liste (18)               229
         durée hors liste (48)               104
         durée hors liste (6)                148
         durée hors liste (9)                 64

Échantillon de 20 lignes :
rule_id       table        id                                     valeur                         famille
 RQ-001 crm_contrat CT0020055 2021-12-18 00:00:00 <= 2021-12-20 00:00:00 date de fin antérieure au début
 RQ-001 crm_contrat CT0000349

In [10]:
if COUCHE == "BRONZE":
    nature, ct = DONNEES["crm_contrat"]
    c = ct[ct.age_debut.notna()].copy()
    ref = pd.Timestamp(DATE_REF)
    c["age_reference"] = age_a(pd.Series(ref, index=c.index), c.date_naissance)
    mineur_signature, mineur_reference = c.age_debut < 18, c.age_reference < 18
    print(f"contrats avec souscripteur connu : {len(c)} | mineurs à la signature : {int(mineur_signature.sum())} | "
          f"mineurs à la date de référence : {int(mineur_reference.sum())} | verdict qui change : {int((mineur_signature != mineur_reference).sum())}")
    print(c[mineur_signature != mineur_reference][["contrat_id", "souscripteur_id", "date_naissance", "date_debut", "age_debut", "age_reference"]].head(10).to_string(index=False))

contrats avec souscripteur connu : 92996 | mineurs à la signature : 301 | mineurs à la date de référence : 157 | verdict qui change : 144
contrat_id souscripteur_id date_naissance date_debut  age_debut  age_reference
 CT0000387         A-22244     1996-09-28 2014-09-01       17.0             30
 CT0001356          A-2454     1998-10-17 2016-02-28       17.0             27
 CT0002683         S-21318     1999-07-06 2017-03-05       17.0             27
 CT0002784         A-30785     2000-07-18 2017-03-27       16.0             26
 CT0003211         H-15141     2001-03-23 2017-06-22       16.0             25
 CT0004249         S-14454     2001-12-29 2018-01-06       16.0             24
 CT0005613         H-30584     2001-02-16 2018-05-14       17.0             25
 CT0006254          A-8673     2000-12-29 2018-07-13       17.0             25
 CT0009140         A-13520     2002-12-12 2019-04-03       16.0             23
 CT0010269         A-24682     2002-12-17 2019-07-19       16.0         

In [11]:
n_debut = int(mineur_signature.sum())
n_ref = int(mineur_reference.sum())
print("| Date de référence | Définition de l'âge | Lignes en échec | Preuve |")
print("|---|---|---|---|")
print(f"| Date de début du contrat | âge en années révolues entre la date de naissance du souscripteur et la date de début du contrat | {fmt(n_debut)} sur {fmt(len(c))} | `notebooks/S2_Fatine.ipynb`, cellules 8 et 9 |")
print(f"| Date du jour (remplacée par la date de référence du lot, {DATE_REF}) | âge en années révolues entre la date de naissance et {DATE_REF} | {fmt(n_ref)} sur {fmt(len(c))} | `notebooks/S2_Fatine.ipynb`, cellules 8 et 9 |")

| Date de référence | Définition de l'âge | Lignes en échec | Preuve |
|---|---|---|---|
| Date de début du contrat | âge en années révolues entre la date de naissance du souscripteur et la date de début du contrat | 301 sur 92 996 | `notebooks/S2_Fatine.ipynb`, cellules 8 et 9 |
| Date du jour (remplacée par la date de référence du lot, 2026-10-01) | âge en années révolues entre la date de naissance et 2026-10-01 | 157 sur 92 996 | `notebooks/S2_Fatine.ipynb`, cellules 8 et 9 |


In [12]:
for m in mesures_ge:
    mesurer(COUCHE, m["rule_id"], m["table"], m["testees"], m["echecs"])
lecture = mesures(COUCHE)
lecture["verdict"] = ["conforme" if t >= s else "sous le seuil" for t, s in zip(lecture.taux_conformite.astype(float), lecture.seuil_min.astype(float))]
print(lecture.to_string(index=False))
print("\nSynthèse par dimension (moyenne des taux de la dernière mesure de chaque règle) :")
print(lire("SELECT dimension, taux_moyen FROM dq.dq_kpi_dimension WHERE batch_id = :b AND couche = :c ORDER BY taux_moyen", b=BATCH, c=COUCHE).to_string(index=False))

couche rule_id      severite      table_cible  nb_testees  nb_echecs taux_conformite seuil_min       verdict
BRONZE  RQ-001     BLOQUANTE      crm_contrat      100000        800          99.200    100.00 sous le seuil
BRONZE  RQ-002     BLOQUANTE      crm_contrat      100000        301          99.699    100.00 sous le seuil
BRONZE  RQ-003 AVERTISSEMENT       crm_client       49898       5989          87.998     98.00 sous le seuil
BRONZE  RQ-005 AVERTISSEMENT       crm_client       95437       1456          98.474     98.00      conforme
BRONZE  RQ-007     BLOQUANTE       crm_client      100000       3050          96.950    100.00 sous le seuil
BRONZE  RQ-009     BLOQUANTE       crm_client      100000        200          99.800    100.00 sous le seuil
BRONZE  RQ-010     BLOQUANTE      crm_contrat      100000        600          99.400    100.00 sous le seuil
BRONZE  RQ-020     BLOQUANTE      cif_clients       40000          0         100.000    100.00      conforme
BRONZE  RQ-020     

In [13]:
FAMILLE_DIMENSION = {"Unicité": "Unicité", "Complétude": "Complétude", "Complétude, intégrité": "Complétude", "Traçabilité": "Complétude",
                     "Validité format": "Validité", "Validité domaine": "Validité", "Validité métier": "Validité",
                     "Intégrité référentielle": "Validité", "Cohérence": "Cohérence"}
dims = lire("SELECT rule_id, dimension FROM dq.regle")
synth = lecture.merge(dims, on="rule_id")
synth["famille"] = synth.dimension.map(FAMILLE_DIMENSION).fillna(synth.dimension)
synth["taux"] = synth.taux_conformite.astype(float)
print("| Dimension | Règles | Taux moyen | Règle la plus faible | Constat |")
print("|---|---|---|---|---|")
for fam in ["Unicité", "Complétude", "Validité", "Cohérence"]:
    s = synth[synth.famille == fam]
    if s.empty:
        print(f"| {fam} | aucune règle mesurée | – | – | … |")
        continue
    pire = s.loc[s.taux.idxmin()]
    regles = ", ".join(sorted(s.rule_id.unique()))
    print(f"| {fam} | {regles} | {s.taux.mean():.2f} % | {pire.rule_id} ({pire.table_cible}, {pire.taux:.3f} %) | … |")
print("\nTaux moyen toutes règles confondues :", round(synth.taux.mean(), 2), "%")
print("Règles sous le seuil :", synth[synth.taux < synth.seuil_min.astype(float)][["rule_id", "table_cible", "taux", "seuil_min", "severite"]].to_string(index=False))

| Dimension | Règles | Taux moyen | Règle la plus faible | Constat |
|---|---|---|---|---|
| Unicité | RQ-020 | 100.00 % | RQ-020 (cif_clients, 100.000 %) | … |
| Complétude | RQ-003, RQ-021, RQ-022 | 99.20 % | RQ-003 (crm_client, 87.998 %) | … |
| Validité | RQ-002, RQ-005, RQ-007, RQ-009, RQ-010 | 98.86 % | RQ-007 (crm_client, 96.950 %) | … |
| Cohérence | RQ-001 | 99.20 % | RQ-001 (crm_contrat, 99.200 %) | … |

Taux moyen toutes règles confondues : 99.26 %
Règles sous le seuil : rule_id table_cible   taux seuil_min      severite
 RQ-001 crm_contrat 99.200    100.00     BLOQUANTE
 RQ-002 crm_contrat 99.699    100.00     BLOQUANTE
 RQ-003  crm_client 87.998     98.00 AVERTISSEMENT
 RQ-007  crm_client 96.950    100.00     BLOQUANTE
 RQ-009  crm_client 99.800    100.00     BLOQUANTE
 RQ-010 crm_contrat 99.400    100.00     BLOQUANTE
